# 06 · Your own data, methods and metrics

The pipeline is built to be extended. The datasets, the unmixing methods, the
selection metrics and the pair assignment strategies are registries: dictionaries
that map a name to an implementation. Add an entry, and the configs can name it.

| To add | Registry | File |
|---|---|---|
| a dataset format | `READERS` | `src/loading/load.py` |
| a selection metric | `METRICS` | `src/unmixing/metrics.py` |
| an unmixing method | `METHODS` | `src/unmixing/models.py` |
| a pair assignment strategy | `STRATEGIES` | `src/unmixing/assignment.py` |

This notebook adds one of each and runs them. Here they are registered from the
notebook so you can try them out; in your own project, put the code in the file
named above and add the entry to the registry there, so `main.py` and saved runs
find it too.

In [1]:
import dataclasses
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from cedalion.sigdecomp.multimodal.cca import CCA

from src.config import SubjectConfig, load_experiment, load_subject
from src.loading import READERS, check_data_format, load_recordings, load_rest
from src.loading.files import read_fif, read_snirf
from src.log import setup_logging
from src.preprocessing import preprocess_recordings
from src.pruning import prune_recordings
from src.types import PairLabels, RawRecording
from src.unmixing import METRICS, find_hands, selection_report
from src.unmixing.assignment import STRATEGIES, Strategy, TrainingPairs
from src.unmixing.metrics import Metric
from src.unmixing.models import METHODS, Method, check_config
from src.unmixing.select import select_and_score

logger = setup_logging()
warnings.filterwarnings("ignore", message=".*encountered in matmul", category=RuntimeWarning)

## 1. A new dataset format

A **reader** is the only part of the pipeline that knows how a dataset is laid out on
disk. It turns the files into `RawRecording` objects (notebook 01), and everything
after loading works on those, whatever the dataset.

Say your data is one EEG file (FIF) and one fNIRS file (SNIRF) per run, one folder per
recording session, and a separate rest recording:

```
<paths.data>/<group>/<run>/eeg_raw.fif
<paths.data>/<group>/<run>/fnirs.snirf
<paths.rest_eeg>, <paths.rest_fnirs>
```

A reader provides `REQUIRED_PATHS` (the `paths` keys it needs in the subject config),
`check` (fail early if the layout is wrong), `read_recordings` (every run of the
subject's groups) and `read_rest` (the recording pruning judges the channels on, with
one stim event marking its rest period). `src/loading/files.py` already reads FIF,
SNIRF and MAT files.

In [2]:
class FifSnirfReader:
    """
    Reader for one FIF and one SNIRF file per run, in one folder per run:

        <paths.data>/<group>/<run>/eeg_raw.fif
        <paths.data>/<group>/<run>/fnirs.snirf

    The rest recording is paths.rest_eeg and paths.rest_fnirs. The data has no
    ground truth.
    """
    REQUIRED_PATHS = ("data", "rest_eeg", "rest_fnirs")

    def check(self, subject: SubjectConfig):
        """Raise if a group has no run folder with both files."""
        for group in subject.groups:
            if not self._run_dirs(subject, group):
                raise FileNotFoundError(f"Group {group!r}: no run folder with eeg_raw.fif and "
                                        f"fnirs.snirf in {subject.paths['data'] / group}.")

    def read_recordings(self, subject: SubjectConfig) -> list[RawRecording]:
        """Every run of every group in `subject.groups`, in that order."""
        return [self._read(run_dir / "eeg_raw.fif", run_dir / "fnirs.snirf", group, run_dir.name)
                for group in subject.groups for run_dir in self._run_dirs(subject, group)]

    def read_rest(self, subject: SubjectConfig) -> RawRecording:
        """The rest recording."""
        return self._read(subject.paths["rest_eeg"], subject.paths["rest_fnirs"], "rest", "rest")

    @staticmethod
    def _run_dirs(subject: SubjectConfig, group: str) -> list[Path]:
        folder = subject.paths["data"] / group
        return sorted(d for d in folder.glob("*")
                      if (d / "eeg_raw.fif").is_file() and (d / "fnirs.snirf").is_file())

    @staticmethod
    def _read(eeg_path: Path, fnirs_path: Path, group: str, run: str) -> RawRecording:
        eeg = read_fif(eeg_path)
        fnirs, geo3d, stim = read_snirf(fnirs_path)
        return RawRecording(group=group, run=run, eeg=eeg, fnirs=fnirs, geo3d=geo3d, stim=stim)


READERS["fif_snirf"] = FifSnirfReader()
print(f"Known formats: {sorted(READERS)}")

Known formats: ['buccino', 'fif_snirf', 'synthetic']


To try the reader, the next cell lays out the files of the tutorial's synthetic
subject in this format, as links under `data/own_format/` (three sessions, to keep
it quick). With your own data, skip it: your files are already there.

In [3]:
source = load_subject(PROJECT_ROOT / "configs/subjects/synthetic/sub01/sub01_snr_fnirs-5_eeg5.yaml")
groups = ["sitting_pre_rest_1", "sitting_pre_rest_2", "sitting_post_rest_1"]
own = PROJECT_ROOT / "data" / "own_format"
for rec_dir in sorted(source.paths["data"].glob(f"{source.subject_id}_*/run_*")):
    group = rec_dir.parent.name.removeprefix(f"{source.subject_id}_")
    if group not in groups:
        continue
    target = own / group / rec_dir.name
    target.mkdir(parents=True, exist_ok=True)
    for name, pattern in (("eeg_raw.fif", "*_raw.fif"), ("fnirs.snirf", "*_amp_*.snirf")):
        link = target / name
        if not link.exists():
            link.symlink_to(next(rec_dir.glob(pattern)))
print(f"{len(list(own.glob('*/*')))} runs in {own.relative_to(PROJECT_ROOT)}")

9 runs in data/own_format


The subject config names the new format and its paths. In a project it lives in
`configs/subjects/<format>/<subject_id>/`; here it is written next to the data.

In [4]:
subject_yaml = own / "my_subject.yaml"
subject_yaml.write_text(f"""\
subject_id: my_subject
format: fif_snirf
paths:
  data: data/own_format
  rest_eeg: {source.paths['rest_eeg'].relative_to(PROJECT_ROOT)}
  rest_fnirs: {source.paths['rest_fnirs'].relative_to(PROJECT_ROOT)}
montage:
  eeg:
    standard: standard_1020
  fnirs:
    recording: true
groups: [{', '.join(groups)}]
test_group: sitting_post_rest_1
trial_classes:
  right: [right]
  left: [left]
""")
subject = load_subject(subject_yaml)
print(subject_yaml.read_text())

subject_id: my_subject
format: fif_snirf
paths:
  data: data/own_format
  rest_eeg: data/background_recordings/sub01/sub01_sitting_pre_rest_1-raw.fif
  rest_fnirs: data/background_recordings/sub01/sub01_sitting_pre_rest_1.snirf
montage:
  eeg:
    standard: standard_1020
  fnirs:
    recording: true
groups: [sitting_pre_rest_1, sitting_pre_rest_2, sitting_post_rest_1]
test_group: sitting_post_rest_1
trial_classes:
  right: [right]
  left: [left]



From here on it is the ordinary pipeline: `check_data_format` finds the reader by the
format name, and loading, pruning and preprocessing run unchanged.

In [5]:
cfg = load_experiment(PROJECT_ROOT / "configs/experiments/tutorial.yaml")

check_data_format(subject)
recordings = load_recordings(subject)
pruned, bad = prune_recordings(recordings, load_rest(subject), cfg.pruning)
recordings = preprocess_recordings(pruned, cfg.preprocessing)
print(f"{len(recordings)} runs loaded, pruned and preprocessed; ground truth: "
      f"{'yes' if recordings[0].ground_truth else 'no'}")

9 runs loaded, pruned and preprocessed; ground truth: no


## 2. A new selection metric

A metric turns the validation scores of one hyperparameter combination into one
number, higher is better. It is a `Metric` with the quantities it averages over the
folds (one value per fold each), whether to average them through Fisher's z (for
correlations), and whether it needs ground truth.

This one selects by the *weaker* hand: the lower of the two pairs' canonical
correlations, so a setting is only good if both hands' pairs hold up.

In [6]:
METRICS["canonical_correlation_min"] = Metric(
    lambda scores: {"min": min(scores.canonical_r.values())},
    fisher=True, needs_ground_truth=False)
print(f"Known metrics: {sorted(METRICS)}")

Known metrics: ['canonical_correlation_avg', 'canonical_correlation_min', 'isi_avg', 'joint_avg']


## 3. A new unmixing method

A method is a `Method` entry: the model class, the hyperparameters its grid may
contain, the EEG feature it needs (`bandpower` or `bandpassed`), and whether it is
temporal (finds a time shift per component) or structured (takes the graph
Laplacians). Any class with the `fit` and `transform` of the cedalion models works.

Here the class is cedalion's CCA, renamed, to show the mechanics. Replace it with your
own model. It needs a grid too, here an empty one since it has no hyperparameters.

In [7]:
class MyCCA(CCA):
    """Your own model goes here; this one is cedalion's CCA unchanged."""


METHODS["MyCCA"] = Method(MyCCA)
print(f"Known methods: {sorted(METHODS)}")

Known methods: ['CCA', 'ElasticNetCCA', 'ElasticNetTCCA', 'MyCCA', 'RidgeCCA', 'SparseCCA', 'StructuredSparseCCA', 'StructuredSparseTCCA', 'mSPoC', 'tCCA']


## 4. A new pair assignment strategy

A strategy gives each hand its own pair, looking at the training runs only. It is a
function with the signature `(pairs, hands, cfg) -> PairLabels`, where `pairs`
(`TrainingPairs`) holds the fitted model, the training data and every pair's
components. This one simply gives the hands the strongest pairs in order.

In [8]:
def strongest_pairs(pairs: TrainingPairs, hands: list[str], cfg) -> PairLabels:
    """
    Give the hands the strongest pairs in order: pair 0 to the first hand, pair 1 to
    the second.

    Parameters
    ----------
    pairs : TrainingPairs
    hands : list[str]
    cfg : AssignmentConfig

    Returns
    -------
    PairLabels
    """
    return PairLabels(pairs={hand: k for k, hand in enumerate(hands)}, score=np.nan)


STRATEGIES["strongest_pairs"] = Strategy(strongest_pairs, needs_ground_truth=False)
print(f"Known strategies: {sorted(STRATEGIES)}")

Known strategies: ['ground_truth', 'motor_channels', 'strongest_pairs', 'task_regressors']


## 5. Using them together

An experiment config would now just name them:

```yaml
unmixing:
  methods: [CCA, MyCCA]
  metrics: [canonical_correlation_avg, canonical_correlation_min]
  assignment: { method: strongest_pairs }
```

Here the same is done by changing the loaded tutorial config (its `grid_overrides`
are for methods not used here, so they are cleared). `n_jobs: 1` keeps the
fits in this notebook's process: worker processes import `src` afresh and would not
see what the notebook registered. Registered in `src`, the new entries work with any
`n_jobs`. `check_config` validates the result, as every run does before loading data.

In [9]:
unmixing = dataclasses.replace(
    cfg.unmixing,
    methods=("CCA", "MyCCA"),
    metrics=("canonical_correlation_avg", "canonical_correlation_min"),
    assignment=dataclasses.replace(cfg.unmixing.assignment, method="strongest_pairs"),
    grids={**cfg.unmixing.grids, "MyCCA": {}},
    grid_overrides={},
    n_jobs=1)
check_config(unmixing, cfg.preprocessing.eeg_feature)

hands = find_hands(recordings, subject.trial_classes, cfg.hand)
for method in unmixing.methods:
    selections, _, _ = select_and_score(recordings, subject.test_group, method, unmixing,
                                        hands, logger=logger,
                                        trial_classes=subject.trial_classes)
    for s in selections:
        print(selection_report(s))

2026-10-30 18:07:35 [INFO] CCA: grid search, 1 combinations x 2 folds (validation sitting_pre_rest_1, sitting_pre_rest_2)


2026-10-30 18:07:35 [INFO] CCA: grid search done in 0 s


2026-10-30 18:07:35 [INFO] MyCCA: grid search, 1 combinations x 2 folds (validation sitting_pre_rest_1, sitting_pre_rest_2)


2026-10-30 18:07:35 [INFO] MyCCA: grid search done in 0 s


CCA by canonical_correlation_avg: validation (dev) -0.019 with {}
    test sitting_post_rest_1  right_hand pair  0  canonical r +0.05
    test sitting_post_rest_1  left_hand  pair  1  canonical r -0.03
CCA by canonical_correlation_min: validation (dev) -0.053 with {}
    test sitting_post_rest_1  right_hand pair  0  canonical r +0.05
    test sitting_post_rest_1  left_hand  pair  1  canonical r -0.03


MyCCA by canonical_correlation_avg: validation (dev) -0.019 with {}
    test sitting_post_rest_1  right_hand pair  0  canonical r +0.05
    test sitting_post_rest_1  left_hand  pair  1  canonical r -0.03
MyCCA by canonical_correlation_min: validation (dev) -0.053 with {}
    test sitting_post_rest_1  right_hand pair  0  canonical r +0.05
    test sitting_post_rest_1  left_hand  pair  1  canonical r -0.03


`MyCCA` gives the same pairs as `CCA`, as it should, since it is the same model. The
naive `strongest_pairs` rule hardly holds up on the test group (canonical r near 0):
the strongest pairs on the training runs need not be the hands' sources, which is why
the pipeline's own strategies look at the motor channels or the trials instead. The
whole run (`src/experiment.py`, `run`) uses the same registries, so once the entries
are in `src`, `python main.py` runs them with classification, saving and figures.

## Changing the steps themselves

Each step is one call in `run` in `src/experiment.py`, which hands each step its
config section and the previous step's output. To add a step, or replace one with
your own implementation, change that list of calls; no step calls another, so the
rest of the pipeline is unaffected.